In [0]:
import java.util.Properties
import org.apache.spark.sql.DataFrame

// Connection Parameters
val DB_HOST = "dsip4_shared_postgres_db"
val DB_PORT = "5432"
val DB_NAME = "UseCase04_G02"
val DB_USER = "UseCase04_G02"
val DB_PASS = "sA5^kR8!eY3tMx1N"

val jdbcUrl = s"jdbc:postgresql://$DB_HOST:$DB_PORT/$DB_NAME"
val pgDriver = "org.postgresql.Driver"

// JDBC Properties
val connProps = new Properties()
connProps.setProperty("user", DB_USER)
connProps.setProperty("password", DB_PASS)
connProps.setProperty("driver", pgDriver)
// Function to load a table
def loadTable(table: String): DataFrame = {
  spark.read
    .format("jdbc")
    .option("url", jdbcUrl)
    .option("dbtable", table)
    .option("user", DB_USER)
    .option("password", DB_PASS)
    .option("driver", pgDriver)
    .load()
}
// Load table
val rawDF = loadTable("ops_20231101_012920_allnw")
val rawCount = rawDF.count()



println(s"Connected to: $jdbcUrl")
println(rawDF.count())
rawDF.select("line")
.show()

In [1]:
val newRawDF0 = spark.read.text("/data/uc04/OPS_20231101_012920_ALLNW.dat")


In [2]:
val newRawDF1 = spark.read.text("/data/uc04/OPS_20231102_000000_ALLNW.dat")
val newRawDF2 = spark.read.text("/data/uc04/OPS_20231103_000000_ALLNW.dat")
val newRawDF3 = spark.read.text("/data/uc04/OPS_20231104_000000_ALLNW.dat")
val newRawDF4 = spark.read.text("/data/uc04/OPS_20231105_000000_ALLNW.dat")
val newRawDF5 = spark.read.text("/data/uc04/OPS_20231106_000000_ALLNW.dat")
val newRawDF6 = spark.read.text("/data/uc04/OPS_20231107_000000_ALLNW.dat")
val newRawDF7 = spark.read.text("/data/uc04/OPS_20231108_000000_ALLNW.dat")

In [3]:
val rawDF00 =
  newRawDF0
    .union(newRawDF1)
    .union(newRawDF2)
    .union(newRawDF3)
    .union(newRawDF4)
    .union(newRawDF5)
    .union(newRawDF6)
    .union(newRawDF7)
val rawDF = rawDF00.withColumnRenamed("value", "line")
val rawCount = rawDF.count()
//rawDF.show(false)

In [4]:
//team 1 DB
import java.util.Properties
import org.apache.spark.sql.DataFrame

// Connection Parameters
val DB_HOST = "dsip_shared_postgres_db"
val DB_PORT = "5432"
val DB_NAME = "UseCase04"
val DB_USER = "dsip4"
val DB_PASS = "Dh1N7834"

val jdbcUrl = s"jdbc:postgresql://$DB_HOST:$DB_PORT/$DB_NAME"
val pgDriver = "org.postgresql.Driver"

// JDBC Properties
val connProps = new Properties()
connProps.setProperty("user", DB_USER)
connProps.setProperty("password", DB_PASS)
connProps.setProperty("driver", pgDriver)
// Function to load a table
def loadTable(table: String): DataFrame = {
  spark.read
    .format("jdbc")
    .option("url", jdbcUrl)
    .option("dbtable", table)
    .option("user", DB_USER)
    .option("password", DB_PASS)
    .option("driver", pgDriver)
    .load()
}
// Load table
val rawDF = loadTable("ops_20231101_012920_allnw")
val rawCount = rawDF.count()


println(s"Connected to: $jdbcUrl")
//rawDF
rawDF.createOrReplaceTempView("dd")

In [5]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types.{IntegerType, LongType, TimestampType}
import spark.implicits._


    println("\n[STEP 2] Parsing fixed-width fields & cleaning data...")

    // ── 2a. Filter out Header ("1" at position 1) and Trailer ("@") rows ──
    val headerType = trim(substring(col("line"), 1, 3))
    val traillerType = trim(substring(col("line"), 1, 1))

    val dataRowsDF = rawDF
      .withColumn("headerType", headerType)
      .withColumn("traillerType", traillerType)
      .filter(col("headerType") =!= "101" && col("traillerType") =!= "@")
      
    val filteredCount = dataRowsDF.count()
    println(s"   ✓ Filtered headers/trailers: $filteredCount data rows remain " +
      s"(removed ${rawCount - filteredCount} control rows).")
      
    // ── 2b. Extract every field per the exact byte-offset dictionary ────────
    
    val parsedDF = dataRowsDF
      // ── Primary Keys & Identification ──
      .withColumn("pan",              trim(substring(col("line"),   1, 19)))
      .withColumn("system_trace",     trim(substring(col("line"),  96,  6)))
      .withColumn("svfe_trace",       trim(substring(col("line"), 112,  9)))

      // ── Timeline Core ──
      .withColumn("svfe_time_raw",    trim(substring(col("line"), 126,  6)))  // hhmmss
      .withColumn("svfe_date_raw",    trim(substring(col("line"), 132,  4)))  // MMDD
      .withColumn("client_txn_dt",    trim(substring(col("line"),  62, 10)))  // MMDDhhmmss

      // ── Diagnostic Core ──
      .withColumn("network_resp_code",  trim(substring(col("line"), 312, 2)))  // F44
      .withColumn("svfe_resp_code",     trim(substring(col("line"), 780, 3)))  // F66
      .withColumn("reversal_flag",      trim(substring(col("line"), 111, 1)).cast(IntegerType))  // F12
      .withColumn("completion_status",  trim(substring(col("line"), 879, 1)).cast(IntegerType))  // F76
      .withColumn("stood_in_for",       trim(substring(col("line"), 880, 1)).cast(IntegerType))  // F77
      .withColumn("issuer_posted",      trim(substring(col("line"), 881, 1)).cast(IntegerType))  // F78

      // ── Scoping & Routing ──
      .withColumn("acquiring_inst",   trim(substring(col("line"), 244, 11)))  // F39
      .withColumn("terminal_type",    trim(substring(col("line"), 874,  5)))  // F75
      .withColumn("issuing_inst",     trim(substring(col("line"), 890, 11)))  // F81
      .withColumn("BER",              trim(substring(col("line"),1179 , 7)))
      .withColumn("Univers",              trim(substring(col("line"),102 , 9)).cast(IntegerType))

      // Drop the raw line — no longer needed
      .drop("line", "headerType", "traillerType")

    // ── 2c. Construct the unified server_timestamp ──────────────────────────

    val withTimestampDF = parsedDF
      .withColumn(
        "server_timestamp_str",
        concat(lit("2023"), col("svfe_date_raw"), col("svfe_time_raw"))
      )
      .withColumn(
        "server_timestamp",
        to_timestamp(col("server_timestamp_str"), "yyyyMMddHHmmss")
      )
      .withColumn(
        "client_timestamp_str",
        concat(lit("2023"), col("client_txn_dt"))
      )
      .withColumn(
        "client_timestamp",
        to_timestamp(col("client_timestamp_str"), "yyyyMMddHHmmss")
      )
      

    // ── 2d. Drop rows where the timestamp failed to parse (null) ────────────
    val validTimestampDF = withTimestampDF
      .filter(col("server_timestamp").isNotNull)
    val invalidTsCount = filteredCount - validTimestampDF.count()
    if (invalidTsCount > 0) {
      println(s"   Dropped $invalidTsCount rows with un-parseable timestamps.")
    }

    // ── 2e. Deduplication on (PAN, System Trace, SVFE Trace, Timestamp) ─────

    /*val dedupedDF = validTimestampDF
    
      .dropDuplicates(Seq("univers","reversal_flag"))
      
    val dedupCount  = dedupedDF.count()
    val dupsRemoved = validTimestampDF.count() - dedupCount
    println(s"   ✓ Deduplication complete: $dedupCount unique transactions " +
      s"(removed $dupsRemoved duplicates).")*/


In [6]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types.{IntegerType, LongType, TimestampType}
import spark.implicits._

 
    println("\n[STEP 3] Computing inter-transaction time gaps (Δt)...")

    val globalWindow = Window.orderBy("server_timestamp")

    //val withDeltaDF = dedupedDF
    val withDeltaDF = validTimestampDF
      .withColumn("prev_timestamp", lag("server_timestamp", 1).over(globalWindow))
      .withColumn(
        "delta_t_seconds",
        when(col("prev_timestamp").isNotNull,
          // unix_timestamp returns Long seconds; difference gives Δt in seconds
          unix_timestamp(col("server_timestamp")) - unix_timestamp(col("prev_timestamp"))
        ).otherwise(lit(0L))
      )

    //println(s"   ✓ Δt (delta_t_seconds) computed for $dedupCount transactions.")

    // Quick sanity check — show distribution of gaps
    /*println("\n   ── Δt Distribution Summary ──")
    withDeltaDF.select("delta_t_seconds").describe().show()
    withDeltaDF
  .filter(col("delta_t_seconds") === 13648)
  .select("pan", "client_txn_dt","server_timestamp","prev_timestamp", "delta_t_seconds")
  .show(false)
  withDeltaDF
  .filter(col("server_timestamp") === "2023-10-30 20:14:34")
  .select("pan", "client_txn_dt","server_timestamp","prev_timestamp", "delta_t_seconds")
  .show(false)*/

In [7]:
cleanedDF.write
  .format("jdbc")
  .option("url", jdbcUrl)
  .option("dbtable", "withDeltaDF")
  .option("user", DB_USER)
  .option("password", DB_PASS)
  .option("driver", pgDriver)
  .mode("append")   // or "append"
  .save()

In [8]:
import java.util.Properties
import org.apache.spark.sql.DataFrame

// Connection Parameters
val DB_HOST = "dsip4_shared_postgres_db"
val DB_PORT = "5432"
val DB_NAME = "UseCase04_G02"
val DB_USER = "UseCase04_G02"
val DB_PASS = "sA5^kR8!eY3tMx1N"

val jdbcUrl = s"jdbc:postgresql://$DB_HOST:$DB_PORT/$DB_NAME"
val pgDriver = "org.postgresql.Driver"

// JDBC Properties
val connProps = new Properties()
connProps.setProperty("user", DB_USER)
connProps.setProperty("password", DB_PASS)
connProps.setProperty("driver", pgDriver)

In [9]:
var cleanedDF = spark.read
  .format("jdbc")
  .option("url", jdbcUrl)
  .option("dbtable", "withDeltaDF")
  .option("user", DB_USER)
  .option("password", DB_PASS)
  .option("driver", pgDriver)
  .load()
cleanedDF.count()
cleanedDF.show()
// Print the schema to inspect column types mapping


In [10]:
import org.apache.spark.sql.functions._

var cleanedDF =  withDeltaDF.withColumn(
  "UID",
  concat(col("reversal_flag"), col("univers"), col("svfe_date_raw"))
)
cleanedDF.count()

In [11]:
import org.apache.spark.sql.functions._

cleanedDF = cleanedDF.withColumn(
  "UID",
  concat(col("reversal_flag"), col("univers"))
)
cleanedDF.show()

In [12]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types.{IntegerType, LongType, TimestampType}
var FORjoins = rawDF
.withColumn("reversal_flag",      trim(substring(col("line"), 111, 1)).cast(IntegerType))
.withColumn("Univers",              trim(substring(col("line"),102 , 9)).cast(IntegerType))
.withColumn("SVFE_messg_type",              trim(substring(col("line"),121 , 4)).cast(IntegerType))

FORjoins = FORjoins.withColumn(
  "UID",
  concat(col("reversal_flag"), col("univers"))
)
FORjoins.show()

In [13]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types.{IntegerType, LongType, TimestampType}
var FORjoinsNew = rawDF
.withColumn("reversal_flag",      trim(substring(col("line"), 111, 1)).cast(IntegerType))
.withColumn("Univers",              trim(substring(col("line"),102 , 9)).cast(IntegerType))
.withColumn("svfe_date_raw", trim(substring(col("line"), 132,  4) ))
.withColumn("SVFE_messg_type",              trim(substring(col("line"),121 , 4)).cast(IntegerType))

FORjoinsNew = FORjoinsNew.withColumn(
  "UID",
  concat(col("reversal_flag"), col("univers"),col("svfe_date_raw"))
)
//FORjoinsNew.show()

In [14]:
cleanedDF = cleanedDF.join(
  FORjoinsNew.select("UID", "SVFE_messg_type"),
  Seq("UID"),
  "left"
)
cleanedDF.show()

In [15]:
cleanedDF = cleanedDF.drop("svfe_date_raw", "svfe_time_raw", "client_txn_dt","server_timestamp_str","client_timestamp_str")

In [16]:
cleanedDF.write
  .format("jdbc")
  .option("url", jdbcUrl)
  .option("dbtable", "newDataCleaned")
  .option("user", DB_USER)
  .option("password", DB_PASS)
  .option("driver", pgDriver)
  .mode("overwrite")   // or "append"
  .save()

In [17]:
var debug = spark.read
  .format("jdbc")
  .option("url", jdbcUrl)
  .option("dbtable", "dataquality")
  .option("user", DB_USER)
  .option("password", DB_PASS)
  .option("driver", pgDriver)
  .load()
  debug.show()

In [18]:
var cleanedDF123 = spark.read
  .format("jdbc")
  .option("url", jdbcUrl)
  .option("dbtable", "withDeltaDF")
  .option("user", DB_USER)
  .option("password", DB_PASS)
  .option("driver", pgDriver)
  .load()

In [19]:
cleanedDF123.printSchema()
//cleanedDF123.select("svfe_date_raw").show()

In [20]:
cleanedDF123.select("UID","pan","client_timestamp","server_timestamp","delta_t_seconds","network_resp_code","svfe_resp_code","reversal_flag","completion_status","stood_in_for","issuer_posted","terminal_type")
.orderBy(asc("server_timestamp"))
/*.filter(
  col("server_timestamp").between(
    "2023-11-03 16:50:00",
    "2023-11-03 16:54:00"
    ))*/
.show(10000,false)

In [21]:
import org.apache.spark.sql.functions._

// 1. Group the massive dataset into 1-minute windows
val timeBinnedDF = cleanedDF123
  .groupBy(window(col("server_timestamp"), "1 minutes"))
  .agg(count("UID").alias("transaction_count"))
  .select(
    col("window.start").cast("string").alias("time_bin"), 
    col("transaction_count")
  )
  .orderBy("time_bin")


timeBinnedDF.createOrReplaceTempView("traffic_summary")

In [22]:
%pyspark
import pandas as pd
from pyspark.sql import SparkSession

# 1. Connect to Spark and pull your data
spark = SparkSession.builder.getOrCreate()
df = spark.sql("SELECT * FROM traffic_summary").toPandas()

# 2. Set DateTime Index
df['time_bin'] = pd.to_datetime(df['time_bin'])
df.set_index('time_bin', inplace=True)

# 3. Pillar 1: Resample to 1-Minute Bins (Ensuring missing minutes are 0)
# '1min' ensures we have a row for every single minute of the day
tpm = df['transaction_count'].resample('1min').sum().fillna(0).reset_index()
tpm.columns = ['timestamp', 'actual_volume']

# 4. Pillar 2: The Baseline (Simulated for this prototype)
# For the prototype, we assume expected traffic is the rolling average of the previous 60 minutes.
# (If a server drops to 0 after doing 500/min for an hour, it's a crash).
tpm['expected_volume'] = tpm['actual_volume'].rolling(window=60, min_periods=1).mean().shift(1).fillna(0)

# 5. Pillar 3: State Classification
# Let's set our noise threshold (theta) to 10 transactions
theta = 10 

def classify_state(row):
    if row['actual_volume'] > 0:
        return 'ONLINE'
    elif row['actual_volume'] == 0 and row['expected_volume'] <= theta:
        return 'IDLE'
    else:
        return 'OFFLINE'

tpm['system_state'] = tpm.apply(classify_state, axis=1)

# 6. Pillar 4: Availability Calculation
total_minutes = len(tpm)
offline_minutes = len(tpm[tpm['system_state'] == 'OFFLINE'])

availability_percentage = (1 - (offline_minutes / total_minutes)) * 100

print("--- SVFE SWITCH AVAILABILITY REPORT ---")
print(f"Total Minutes Evaluated: {total_minutes}")
print(f"Total Downtime Minutes: {offline_minutes}")
print(f"Estimated Availability: {availability_percentage:.4f}%\n")

# Display the exact minutes the system was down
print("Downtime Incident Log:")
print(tpm[tpm['system_state'] == 'OFFLINE'].head())

In [23]:
%pyspark
import io
import base64

# 1. Save the plot to a temporary memory buffer
buf = io.BytesIO()
plt.savefig(buf, format='png', bbox_inches='tight')
buf.seek(0)

# 2. Convert the image to a Base64 text string
img_str = base64.b64encode(buf.read()).decode('utf-8')
plt.close()

# 3. Force SkyNote to render it as an HTML image tag
html_code = f"<img src='data:image/png;base64,{img_str}' style='max-width:100%; height:auto;'/>"

# Try printing as standard HTML (Jupyter style)
try:
    from IPython.display import HTML, display
    display(HTML(html_code))
except ImportError:
    # Fallback to Zeppelin/SkyNote style
    print(f"%html {html_code}")

In [24]:
%pyspark
import pandas as pd
import plotly.express as px
from pyspark.sql import SparkSession

# 1. Connect to Spark and pull the data
spark = SparkSession.builder.getOrCreate()
df = spark.sql("SELECT * FROM traffic_summary").toPandas()

# 2. Convert to DateTime and set as index
df['time_bin'] = pd.to_datetime(df['time_bin'])
df.set_index('time_bin', inplace=True)

# 3. Force the empty crash voids to show as ZERO (so the line drops)
continuous_traffic = df['transaction_count'].resample('1min').sum().fillna(0).reset_index()

# 4. Create the Interactive Plotly Chart
fig = px.line(
    continuous_traffic, 
    x='time_bin', 
    y='transaction_count', 
    title='SVFE Switch Traffic (Interactive Dashboard)',
    labels={'time_bin': 'Exact Server Time', 'transaction_count': 'Transactions (1-min window)'}
)

# 5. Make the line look professional (Djezzy red/blue theme)
fig.update_traces(line_color='#00529B', line_width=2)
fig.update_layout(hovermode="x unified") # Shows a clean tooltip when you hover

# 6. Render the interactive HTML graph!
# If this just prints an object name, change it to: fig.show(renderer="html")
html_plot = fig.to_html(full_html=False, include_plotlyjs='cdn')

# Force SkyNote to render the HTML
try:
    from IPython.display import HTML, display
    display(HTML(html_plot))
except ImportError:
    # Fallback if standard IPython display fails
    print(f"%html {html_plot}")

In [25]:
%pyspark
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest


df = spark.sql("SELECT * FROM traffic_summary").toPandas()

# Convert time column to actual Datetime and set it as the index
df['time_bin'] = pd.to_datetime(df['time_bin'])
df.set_index('time_bin', inplace=True)

# This resamples the data to exact 1-minute intervals and forces empty minutes to be 0
tpm = df['transaction_count'].resample('1min').sum().fillna(0).reset_index()
tpm.columns = ['timestamp', 'actual_volume']
tpm.set_index('timestamp', inplace=True)



# What was the median traffic over the last 10 minutes?
tpm['plateau_context'] = tpm['actual_volume'].rolling(window=10, min_periods=1).median().shift(1).fillna(0)

# Did we just drop suddenly from the previous minute?
tpm['sudden_drop'] = tpm['actual_volume'].diff().fillna(0)



# THE AI MODEL

features = ['actual_volume', 'plateau_context', 'sudden_drop']
X = tpm[features].fillna(0)

# Train the model (contamination=0.01 because micro-outages are rare)
iso_forest = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
tpm['anomaly_flag'] = iso_forest.fit_predict(X)


# We only want to see true crashes: Flagged by AI AND volume hit 0 AND it was previously busy
micro_outages = tpm[
    (tpm['anomaly_flag'] == -1) & 
    (tpm['actual_volume'] == 0) & 
    (tpm['plateau_context'] > 10) # Using 10 as a noise threshold so it ignores dead nights
]

print("--- DETECTED MIDDAY MICRO-OUTAGES ---")
print(f"Total isolated drops found: {len(micro_outages)}\n")
print(micro_outages[['actual_volume', 'plateau_context', 'sudden_drop']])

In [26]:
%python
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# 1. Assuming 'tpm' is your 1-minute resampled DataFrame from the previous step
# We must engineer "Shape Features" because we only have 1 week of data

# Feature 1: The current volume
# Feature 2: Rate of Change (First Derivative - catches sudden vertical drops)
tpm['rate_of_change'] = tpm['actual_volume'].diff().fillna(0)

# Feature 3: 15-Minute Rolling Variance (Catches the unnatural "flatlines" of synthetic recovery)
tpm['rolling_variance'] = tpm['actual_volume'].rolling(window=15, min_periods=1).var().fillna(0)

# Feature 4: Hour of the Day (Gives the model a tiny bit of daily context)
tpm['hour_of_day'] = tpm.index.hour

# 2. Isolate the features for the ML model
features = ['actual_volume', 'rate_of_change', 'rolling_variance', 'hour_of_day']
X = tpm[features].fillna(0)

# 3. Initialize and Train the Isolation Forest
# contamination=0.05 assumes roughly 5% of your week was anomalous downtime/recovery
iso_forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)

# 4. Predict Anomalies (-1 is Anomaly, 1 is Normal)
tpm['anomaly_flag'] = iso_forest.fit_predict(X)

# Map the results to readable labels
tpm['system_state'] = np.where(tpm['anomaly_flag'] == -1, 'ANOMALY_DETECTED', 'ONLINE')

# View the flagged anomalies
print(tpm[tpm['system_state'] == 'ANOMALY_DETECTED'].head(15))

In [27]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._
val w = Window.orderBy("server_timestamp")
val result = cleanedDF123
  .withColumn(
    "rule",
    when(col("delta_t_seconds") > 300, "RULE_1_MASSIVE_GAP")
  )
  .withColumn(
    "next_is_gap",
    lead(col("rule").isNotNull, 1).over(w)
  )
  .filter(col("rule").isNotNull || col("next_is_gap"))
  .orderBy("server_timestamp")
  .select ("server_timestamp","pan","delta_t_seconds")
  .show(false)

In [28]:
cleanedDF
.select("pan","client_timestamp","server_timestamp","delta_t_seconds","network_resp_code","svfe_resp_code","reversal_flag","completion_status","stood_in_for","issuer_posted","terminal_type")
.orderBy(asc("server_timestamp"))
.show(100,false)

In [29]:
val DB_HOST = "dsip_shared_postgres_db"
val DB_PORT = "5432"
val DB_NAME = "UseCase04"
val DB_USER = "dsip4"
val DB_PASS = "Dh1N7834"

val JDBC_URL = s"jdbc:postgresql://$DB_HOST:$DB_PORT/$DB_NAME"

val connectionProperties = new java.util.Properties()
connectionProperties.setProperty("user", DB_USER)
connectionProperties.setProperty("password", DB_PASS)
connectionProperties.setProperty("driver", "org.postgresql.Driver")

val opsDF = spark.read.jdbc(
  JDBC_URL,
  """(
     SELECT *
     FROM ops_20231101_012920_allnw
     )as opsDF""",
  connectionProperties
)
val refCodeDF = spark.read.jdbc(
  JDBC_URL,
  """(
     SELECT *
     FROM reference_responsecode
     )as refCodeDF""",
  connectionProperties
)
val refCurrencyDF = spark.read.jdbc(
  JDBC_URL,
  """(
     SELECT *
     FROM reference_currency 
     )as refCurrencyDF""",
  connectionProperties
)
val refInstitutionsDF = spark.read.jdbc(
  JDBC_URL,
  """(
     SELECT *
     FROM reference_institutions
     )as refInstitutionsDF""",
  connectionProperties
)
val DictionaryDF = spark.read.jdbc(
  JDBC_URL,
  """(
     SELECT *
     FROM dictionary
     )as DictionaryDF""",
  connectionProperties
)
opsDF.show(false)
refCodeDF.show(1000,false)
refCodeDF.createOrReplaceTempView("ref_svfe_code")
refCurrencyDF.show
refCurrencyDF.createOrReplaceTempView("refCurrency")
refInstitutionsDF.show()
refInstitutionsDF.createOrReplaceTempView("refInstitutions")
DictionaryDF.show()
DictionaryDF.createOrReplaceTempView("Dictionary")

In [30]:
%sql
select * from ref_svfe_code;
select * from refCurrency;
select * from refInstitutions;
select * from Dictionary;

In [31]:
val tablesDF = spark.read
  .jdbc(jdbcUrl, "information_schema.tables", connProps)
  .filter("table_schema = 'private'") //
  .select("table_name")
  tablesDF.show()

In [32]:
val tablesDF = spark.read
  .jdbc(jdbcUrl, "dictionary", connProps)
  tablesDF.show()
  
  

In [33]:
%spark


In [ ]:
%pyspark
import pandas as pd

# Get the traffic summary data
df = spark.sql("SELECT * FROM traffic_summary").toPandas()
df['time_bin'] = pd.to_datetime(df['time_bin'])
df.set_index('time_bin', inplace=True)

# Resample to 1-minute bins to ensure we have every minute, filling missing with 0
tpm = df['transaction_count'].resample('1min').sum().fillna(0).reset_index()
tpm.columns = ['timestamp', 'actual_volume']
tpm.set_index('timestamp', inplace=True)

# Rule-Based Detection: Find periods where actual_volume == 0 for more than 3 consecutive minutes
# This perfectly captures night-time gaps and micro-outages without the ML noise threshold.
is_zero = tpm['actual_volume'] == 0
zero_groups = (is_zero != is_zero.shift()).cumsum()

# Calculate streaks of 0s
zero_streaks = tpm[is_zero].groupby(zero_groups).apply(
    lambda x: pd.Series({
        'gap_start': x.index.min(),
        'gap_end': x.index.max(),
        'duration_minutes': len(x)
    })
)

# Filter for gaps longer than 3 minutes
gaps = zero_streaks[zero_streaks['duration_minutes'] > 3].copy()

print("--- DETECTED GAPS (>3 Minutes) ---")
print(f"Total exact outages found: {len(gaps)}\n")

if not gaps.empty:
    # Format timestamps for readability
    gaps['gap_start'] = gaps['gap_start'].dt.strftime('%Y-%m-%d %H:%M:%S')
    gaps['gap_end'] = gaps['gap_end'].dt.strftime('%Y-%m-%d %H:%M:%S')
    print(gaps.to_string(index=False))